# V2: Adaptive Deep Research

This notebook evolves the V1 static multi-agent workflow into a pure-Python,
adaptive deep research system.

## V2 Learning Goals

- Convert user requests into structured research briefs.
- Replace the static research plan with an observation-driven Supervisor loop.
- Run context-isolated Research Workers on focused subtopics.
- Preserve raw evidence while compressing results for Supervisor decisions.
- Add bounded parallel research with `asyncio`.
- Reuse the V1 Writer-Editor reflection and deterministic report validation.

> **Status:** Scoping is implemented. Adaptive supervision and isolated research
> workers are the next development stage. Reusable V1 components remain below as
> an executable reference while V2 replaces the static orchestration incrementally.

# 0. Setup

## 0.1 Imports

In [ ]:
import json
import os
import platform
import re
import sys
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from pydantic import ValidationError

from agentic_deepresearch.schemas import (
    ClarificationDecision,
    ConversationMessage,
    PlanStep,
    ResearchBrief,
    ScopeOutcome,
    TaskPlan,
)
from agentic_deepresearch.tools import (
    arxiv_search_tool,
    arxiv_tool_def,
    tavily_search_tool,
    tavily_tool_def,
)

## 0.2 Runtime Configuration

In [ ]:
EXPECTED_PYTHON_VERSION = (3, 12)

env_file = find_dotenv(usecwd=True)
if not env_file:
    raise RuntimeError("Could not locate the project .env file.")

ENV_PATH = Path(env_file).resolve()
PROJECT_ROOT = ENV_PATH.parent

if sys.version_info[:2] != EXPECTED_PYTHON_VERSION:
    raise RuntimeError("Python 3.12 is required. Start Jupyter with `uv run jupyter lab`.")

load_dotenv(dotenv_path=ENV_PATH, override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
MODEL_NAME = os.getenv("MODEL_NAME")

if not openai_api_key:
    raise RuntimeError("OPENAI_API_KEY is not configured in .env.")

if not MODEL_NAME:
    raise RuntimeError("MODEL_NAME is not configured in .env.")

client = OpenAI(api_key=openai_api_key)

print(f"[SETUP] Python: {platform.python_version()}")
print(f"[SETUP] Executable: {sys.executable}")
print(f"[SETUP] Project directory: {PROJECT_ROOT}")
print(f"[SETUP] .env found: {ENV_PATH.exists()}")
print(f"[SETUP] Model: {MODEL_NAME}")
print(f"[SETUP] Tavily configured: {bool(os.getenv('TAVILY_API_KEY'))}")

## 0.3 Notebook Display Helper

In [ ]:
def show(title: str, content: str) -> None:
    display(Markdown(f"## {title}\n\n{content}"))

# 1. Scoping

## 1.1 Architecture and Data Contracts

Scoping converts an informal conversation into a standalone research contract.
The LLM makes semantic decisions, while ordinary Python owns routing and state.

```text
ConversationMessage[]
        |
        v
ClarificationDecision -- needs clarification --> human answer
        |
        v
ResearchBrief
        |
        v
ScopeOutcome
```

The Pydantic definitions live in `agentic_deepresearch.schemas`. The examples
below validate the contracts without making API calls.

In [ ]:
conversation_example = [
    ConversationMessage(
        role="user",
        content="Research recent developments in reliable LLM agents.",
    )
]

clarification_example = ClarificationDecision(
    needs_clarification=True,
    reason="The requested research period is not specified.",
    question="What time range should the research cover?",
)

try:
    ClarificationDecision(
        needs_clarification=True,
        reason="The requested research period is not specified.",
        question=None,
    )
except ValidationError as exc:
    print(f"Expected validation error: {exc.errors()[0]['msg']}")

print(conversation_example[0].model_dump())
print(clarification_example.model_dump())

In [ ]:
research_brief_example = ResearchBrief(
    research_question=("What recent design patterns improve the reliability of LLM agents?"),
    objective=(
        "Help software engineers compare practical patterns for building "
        "more reliable LLM agent systems."
    ),
    scope_inclusions=[
        "Planning and execution patterns",
        "Tool-use reliability",
        "Reflection and verification",
        "Multi-agent coordination",
    ],
    scope_exclusions=[
        "General prompt engineering unrelated to agent workflows",
        "Non-LLM autonomous systems",
    ],
    constraints=[
        "Focus on developments from 2023 onward",
        "Prioritize peer-reviewed papers and primary technical sources",
        "Write for software engineers",
    ],
    deliverable=(
        "A source-backed technical Markdown report with inline citations and a references section."
    ),
    success_criteria=[
        "Identify the major reliability pattern categories",
        "Explain the mechanism and trade-offs of each category",
        "Support important claims with source URLs",
        "Discuss limitations and open research problems",
    ],
)

scope_outcome_example = ScopeOutcome(
    clarification=ClarificationDecision(
        needs_clarification=False,
        reason="The request contains sufficient information for research.",
        question=None,
    ),
    research_brief=research_brief_example,
)

print(scope_outcome_example.model_dump_json(indent=2))

In [ ]:
def build_conversation_input(
    messages: list[ConversationMessage],
) -> list[Any]:
    """Convert a validated conversation into Responses API input items."""

    if not messages:
        raise ValueError("Conversation messages must not be empty.")

    if messages[-1].role != "user":
        raise ValueError("The latest conversation message must be from the user.")

    return [
        {
            "role": message.role,
            "content": message.content,
        }
        for message in messages
    ]

## 1.2 Clarification Agent

The Clarification Agent makes one structured decision: whether a material
ambiguity must be resolved before research begins.

In [ ]:
CLARIFICATION_INSTRUCTIONS = """
You are a clarification agent for a deep research system.

Your responsibility is to determine whether the user's request contains a
material ambiguity that must be resolved before research can begin.

Request clarification only when missing information would substantially change:
- the research topic or central question;
- the intended scope or boundaries;
- the required time period;
- the target audience;
- the expected deliverable.

Do not request clarification for optional preferences that can be handled with
reasonable defaults.

If clarification is required:
- set needs_clarification to true;
- provide a brief operational reason;
- ask exactly one concise question that resolves the most important ambiguity.

If clarification is not required:
- set needs_clarification to false;
- provide a brief operational reason;
- set question to null.

Do not answer the research question.
Do not create a research plan.
Do not include conversational framing or optional follow-up text.
"""

In [ ]:
def clarification_agent(
    messages: list[ConversationMessage],
    model: str = MODEL_NAME,
    verbose: bool = True,
) -> ClarificationDecision:
    """Determine whether the research request requires clarification."""

    input_items = build_conversation_input(messages)

    if verbose:
        print("[CLARIFICATION_AGENT] Starting clarification analysis.")
        print(f"[CLARIFICATION_AGENT] Conversation messages: {len(messages)}")
        print("[CLARIFICATION_AGENT] Model call #1")

    response = client.responses.parse(
        model=model,
        instructions=CLARIFICATION_INSTRUCTIONS,
        input=input_items,
        text_format=ClarificationDecision,
    )

    decision = response.output_parsed

    if decision is None:
        raise RuntimeError("Clarification agent did not return a valid ClarificationDecision.")

    if verbose:
        print("[CLARIFICATION_AGENT] Clarification analysis completed.")
        print(f"[CLARIFICATION_AGENT] Needs clarification: {decision.needs_clarification}")

    return decision

## 1.3 Research Brief Agent

Once the conversation is sufficiently clear, this agent converts it into a
standalone `ResearchBrief`. It does not research, plan, or select tools.

In [ ]:
RESEARCH_BRIEF_INSTRUCTIONS = """
You are a research scoping agent for a deep research system.

Convert the complete user conversation into a standalone ResearchBrief.

The brief must preserve the user's explicit requirements and clarify:
- the central research question;
- the purpose of the research;
- what must be included;
- what must be excluded;
- relevant constraints;
- the expected deliverable;
- observable success criteria.

Follow these rules:
- Use the entire conversation, including the user's clarification answers.
- Preserve explicit requirements without weakening or expanding them.
- Infer only reasonable defaults for optional details.
- Do not invent constraints that materially change the research direction.
- Keep scope inclusions specific and non-overlapping.
- Use an empty list when no explicit scope exclusions exist.
- Write success criteria as research outcomes, not execution steps.
- Produce a standalone brief that can be understood without the conversation.

Do not answer the research question.
Do not perform research.
Do not create a research plan.
Do not assign work to agents.
Do not select tools or sources.
Do not include conversational framing or optional follow-up text.
"""

In [ ]:
def research_brief_agent(
    messages: list[ConversationMessage],
    model: str = MODEL_NAME,
    verbose: bool = True,
) -> ResearchBrief:
    """Convert a scoped conversation into a standalone research brief."""

    input_items = build_conversation_input(messages)
    current_date = datetime.now(UTC).date().isoformat()
    instructions = f"{RESEARCH_BRIEF_INSTRUCTIONS}\n\nCurrent date: {current_date}"

    if verbose:
        print("[RESEARCH_BRIEF_AGENT] Starting brief generation.")
        print(f"[RESEARCH_BRIEF_AGENT] Conversation messages: {len(messages)}")
        print(f"[RESEARCH_BRIEF_AGENT] Current date: {current_date}")
        print("[RESEARCH_BRIEF_AGENT] Model call #1")

    response = client.responses.parse(
        model=model,
        instructions=instructions,
        input=input_items,
        text_format=ResearchBrief,
    )

    research_brief = response.output_parsed

    if research_brief is None:
        raise RuntimeError("Research brief agent did not return a valid ResearchBrief.")

    if verbose:
        print("[RESEARCH_BRIEF_AGENT] Brief generation completed.")
        print(f"[RESEARCH_BRIEF_AGENT] Research question: {research_brief.research_question}")

    return research_brief

## 1.4 Scoping Controller

The controller uses deterministic Python routing: it pauses when clarification
is required and calls the Research Brief Agent only when the request is ready.

In [ ]:
def scoping_controller(
    messages: list[ConversationMessage],
    model: str = MODEL_NAME,
    verbose: bool = True,
) -> ScopeOutcome:
    """Run the scoping workflow until clarification or a research brief."""

    if not messages:
        raise ValueError("Conversation messages must not be empty.")

    if verbose:
        print("[SCOPING_CONTROLLER] Starting scoping workflow.")
        print(f"[SCOPING_CONTROLLER] Conversation messages: {len(messages)}")
        print("[SCOPING_CONTROLLER] Dispatching to clarification_agent.")

    clarification = clarification_agent(
        messages=messages,
        model=model,
        verbose=verbose,
    )

    if clarification.needs_clarification:
        if verbose:
            print("[SCOPING_CONTROLLER] Scoping paused.")
            print("[SCOPING_CONTROLLER] Route: clarification_required")
            print(f"[SCOPING_CONTROLLER] Question: {clarification.question}")

        return ScopeOutcome(
            clarification=clarification,
            research_brief=None,
        )

    if verbose:
        print("[SCOPING_CONTROLLER] Route: research_brief_generation")
        print("[SCOPING_CONTROLLER] Dispatching to research_brief_agent.")

    research_brief = research_brief_agent(
        messages=messages,
        model=model,
        verbose=verbose,
    )

    outcome = ScopeOutcome(
        clarification=clarification,
        research_brief=research_brief,
    )

    if verbose:
        print("[SCOPING_CONTROLLER] Scoping workflow completed.")
        print("[SCOPING_CONTROLLER] Research brief is ready.")

    return outcome

## 1.5 Conversation State Helpers

These helpers create and extend conversation state without mutating the
original message list.

In [ ]:
def create_initial_messages(
    user_request: str,
) -> list[ConversationMessage]:
    """Create a conversation containing the initial user request."""

    user_request = user_request.strip()

    if not user_request:
        raise ValueError("User request must not be empty.")

    return [
        ConversationMessage(
            role="user",
            content=user_request,
        )
    ]


def append_clarification_answer(
    messages: list[ConversationMessage],
    scope_outcome: ScopeOutcome,
    user_answer: str,
) -> list[ConversationMessage]:
    """Return a new conversation containing one clarification exchange."""

    if not messages:
        raise ValueError("Conversation messages must not be empty.")

    if messages[-1].role != "user":
        raise ValueError("The existing conversation must end with a user message.")

    if not scope_outcome.clarification.needs_clarification:
        raise ValueError(
            "A clarification answer cannot be added when clarification is not required."
        )

    clarification_question = scope_outcome.clarification.question

    if clarification_question is None:
        raise RuntimeError("Clarification question is missing from the scope outcome.")

    user_answer = user_answer.strip()

    if not user_answer:
        raise ValueError("Clarification answer must not be empty.")

    return [
        *messages,
        ConversationMessage(
            role="assistant",
            content=clarification_question,
        ),
        ConversationMessage(
            role="user",
            content=user_answer,
        ),
    ]

## 1.6 End-to-End Scoping Demo

The demo simulates a human-in-the-loop clarification cycle. Set
`RUN_SCOPING_DEMO = True` to run it. A complete run makes up to three model
calls, so it is disabled by default.

In [ ]:
RUN_SCOPING_DEMO = False

if RUN_SCOPING_DEMO:
    demo_messages = create_initial_messages("Research recent developments and prepare a report.")
    demo_initial_outcome = scoping_controller(
        messages=demo_messages,
        verbose=True,
    )

    if demo_initial_outcome.clarification.needs_clarification:
        print("\nClarification question:")
        print(demo_initial_outcome.clarification.question)
    else:
        print("\nResearch brief:")
        print(demo_initial_outcome.research_brief.model_dump_json(indent=2))

In [ ]:
if RUN_SCOPING_DEMO:
    demo_user_answer = (
        "Research design patterns developed from 2023 onward that improve "
        "the reliability of LLM agents. Produce a source-backed technical "
        "Markdown report for software engineers."
    )

    demo_messages = append_clarification_answer(
        messages=demo_messages,
        scope_outcome=demo_initial_outcome,
        user_answer=demo_user_answer,
    )

    for message in demo_messages:
        print(f"{message.role}: {message.content}")

In [ ]:
if RUN_SCOPING_DEMO:
    demo_final_outcome = scoping_controller(
        messages=demo_messages,
        verbose=True,
    )

    if demo_final_outcome.clarification.needs_clarification:
        print("\nAdditional clarification question:")
        print(demo_final_outcome.clarification.question)
    else:
        scoping_research_brief = demo_final_outcome.research_brief

        if scoping_research_brief is None:
            raise RuntimeError("Research brief is missing from a completed scope outcome.")

        print("\nResearch brief:")
        print(scoping_research_brief.model_dump_json(indent=2))

# 2. Reusable V1 Components

These components preserve the working V1 baseline while V2 develops adaptive
supervision. They remain visible for comparison, but their standalone API tests
are omitted because V1 already contains the complete course walkthrough.

## 2.1 Tool Infrastructure

The tool registry exposes arXiv and Tavily to research agents through ordinary
Python function dispatch.

In [ ]:
TOOLS = [arxiv_tool_def, tavily_tool_def]

TOOL_REGISTRY = {
    "arxiv_search_tool": arxiv_search_tool,
    "tavily_search_tool": tavily_search_tool,
}


def execute_tool(name: str, arguments: dict) -> Any:
    tool = TOOL_REGISTRY.get(name)

    if tool is None:
        raise ValueError(f"Tool '{name}' not found in the registry.")

    return tool(**arguments)

## 2.2 Static Planner Baseline

In [ ]:
PLANNER_INSTRUCTIONS = """
You are the Planner Agent for a multi-agent deep research workflow.

Available agents and their exact identifiers:
- research_agent: searches the web and arXiv and returns source-backed notes.
- writer_agent: drafts and revises complete Markdown reports.
- editor_agent: reviews drafts and returns actionable feedback.

Create the smallest valid TaskPlan that satisfies the workflow constraints.
Assign exactly one available agent to every step.
Put the action, evidence requirements, and required result directly in each task.
Combine closely related research or writing work instead of creating many small steps.

Include research, one initial draft, exactly one editorial review, and one final revision.
Use at least one research_agent step before the first writer_agent step.
The editor_agent step must be followed immediately by the final writer_agent step.
The final writer_agent step must revise the draft and return the publication-ready report.
Do not add a separate polishing or finalization step after the final revision.
Include only research workflow steps that directly contribute to the final
source-backed report and can be executed by the assigned agents.
Leave report length to the runtime requirements supplied to writer_agent.
Do not execute the plan or include research findings in the plan.
""".strip()

In [ ]:
def validate_task_plan(plan: TaskPlan, max_steps: int) -> None:
    """Validate structural and workflow invariants for a research plan."""

    if max_steps < 4:
        raise ValueError("max_steps must be at least 4 for this workflow.")

    if not plan.steps:
        raise RuntimeError("Planner agent returned a plan with no steps.")

    if len(plan.steps) > max_steps:
        raise RuntimeError(f"Planner returned {len(plan.steps)} steps; maximum is {max_steps}.")

    expected_ids = list(range(1, len(plan.steps) + 1))
    actual_ids = [step.id for step in plan.steps]

    if actual_ids != expected_ids:
        raise RuntimeError(f"Planner step IDs must be consecutive and one-based: {actual_ids!r}")

    route = [step.agent for step in plan.steps]
    research_positions = [index for index, agent in enumerate(route) if agent == "research_agent"]
    writer_positions = [index for index, agent in enumerate(route) if agent == "writer_agent"]
    editor_positions = [index for index, agent in enumerate(route) if agent == "editor_agent"]

    if not research_positions:
        raise RuntimeError("Task plan must include at least one research_agent step.")

    if len(editor_positions) != 1:
        raise RuntimeError("Task plan must include exactly one editor_agent step.")

    editor_position = editor_positions[0]

    if not writer_positions or writer_positions[-1] != len(route) - 1:
        raise RuntimeError("The final plan step must use writer_agent.")

    if editor_position != len(route) - 2:
        raise RuntimeError(
            "The editor_agent step must be immediately before the final writer_agent step."
        )

    draft_writer_positions = [
        position for position in writer_positions if position < editor_position
    ]

    if not draft_writer_positions:
        raise RuntimeError("Task plan must draft the report before editorial review.")

    if not any(position < draft_writer_positions[0] for position in research_positions):
        raise RuntimeError("Research must occur before the initial report draft.")


def planner_agent(
    topic: str,
    model: str = MODEL_NAME,
    max_steps: int = 4,
    verbose: bool = True,
) -> TaskPlan:
    """
    Generate a structured execution plan for a research workflow.

    Args:
        topic: The research topic to investigate.
        model: The language model to use.
        max_steps: Maximum number of executable plan steps.
        verbose: Whether to print concise execution logs.

    Returns:
        A validated task plan with explicit agent assignments.
    """

    topic = topic.strip()

    if not topic:
        raise ValueError("Topic must not be empty.")

    if max_steps < 4:
        raise ValueError("max_steps must be at least 4 for this workflow.")

    if verbose:
        print("[PLANNER_AGENT] Starting planning.")
        print(f"[PLANNER_AGENT] Topic: {topic}")
        print("[PLANNER_AGENT] Model call #1")

    response = client.responses.parse(
        model=model,
        instructions=PLANNER_INSTRUCTIONS,
        input=(
            f"Create a research plan with 4 to {max_steps} steps "
            f"for the following topic:\n\n{topic}"
        ),
        text_format=TaskPlan,
    )

    plan = response.output_parsed

    if plan is None:
        raise RuntimeError(
            f"Planner agent did not return a valid TaskPlan. Raw output: {response.output_text!r}"
        )

    validate_task_plan(plan, max_steps=max_steps)

    if verbose:
        route = " -> ".join(step.agent for step in plan.steps)
        print("[PLANNER_AGENT] Planning completed.")
        print(f"[PLANNER_AGENT] Total steps: {len(plan.steps)}")
        print(f"[PLANNER_AGENT] Agent route: {route}")

    return plan

## 2.3 Research Agent

In [ ]:
RESEARCH_INSTRUCTIONS = """
You are the Research Agent in a multi-agent deep research system.

Execute exactly one research step assigned by the Planner Agent.

Research requirements:

- Focus only on the supplied task step.
- Do not create or modify the task plan.
- Do not answer the user's overall question.
- Use the available tools to collect evidence.
- Use tavily_search_tool for recent developments, documentation, websites,
  articles, and general web information.
- Use arxiv_search_tool for papers, methods, experiments, and benchmarks.
- Use plain semantic keywords for arXiv. Do not use URLs, site operators,
  the word arXiv, or web-style date ranges such as 2020..2026.
- Refine a search query only when its results leave a material evidence gap.
- Prefer 3 to 5 focused searches over many overlapping searches.
- Do not repeat searches that differ only slightly.
- Base factual claims on tool results rather than unsupported memory.
- Do not invent sources, titles, authors, dates, or URLs.
- Preserve source URLs exactly as returned by the tools.
- Prefer primary and academic sources when available.
- Clearly identify incomplete or conflicting evidence.
- Stop once the supplied task can be answered with sufficient relevant evidence.
- After collecting several useful primary and web sources, synthesize the
  findings instead of continuing to search.

Return concise research notes in Markdown using this structure:

## Summary

A concise summary of the research result.

## Findings

Evidence-backed findings with source links.

## Sources

A list of source titles and URLs.

## Limitations

Missing, uncertain, or conflicting information.

Return only the structured research notes. Do not include conversational
text or a user-facing final answer.

Your output will be consumed by later agents through the execution history.
""".strip()

In [ ]:
def research_agent(
    user_request: str,
    step: PlanStep,
    history: list[dict[str, Any]],
    model: str = MODEL_NAME,
    max_tool_rounds: int = 8,
    verbose: bool = True,
) -> str:
    if step.agent != "research_agent":
        raise ValueError(
            f"Task step {step.id!r} is assigned to {step.agent!r}, not 'research_agent'."
        )

    research_request = {
        "user_request": user_request,
        "task": step.task,
        "execution_history": history,
    }

    input_items: list[Any] = [
        {
            "role": "user",
            "content": json.dumps(
                research_request,
                ensure_ascii=False,
            ),
        }
    ]

    model_calls = 0
    tool_rounds = 0
    total_tool_calls = 0
    tool_counts: dict[str, int] = {}

    if verbose:
        print(f"[{step.agent.upper()}] Starting step {step.id}")
        print(f"[{step.agent.upper()}] Task: {step.task}")

    while True:
        tool_budget_exhausted = tool_rounds >= max_tool_rounds
        request_input = input_items

        if tool_budget_exhausted:
            request_input = [
                *input_items,
                {
                    "role": "user",
                    "content": (
                        "The tool budget is exhausted. Do not request more tools. "
                        "Produce the final research notes using the evidence "
                        "already collected."
                    ),
                },
            ]

            if verbose:
                print(f"[{step.agent.upper()}] Tool budget exhausted; forcing final synthesis.")

        model_calls += 1

        if verbose:
            print(f"[{step.agent.upper()}] Model call #{model_calls}")

        response = client.responses.create(
            model=model,
            instructions=RESEARCH_INSTRUCTIONS,
            input=request_input,
            tools=TOOLS,
            tool_choice="none" if tool_budget_exhausted else "auto",
            parallel_tool_calls=False,
        )

        # Preserve all model output items for the next API request.
        input_items.extend(response.output)

        tool_calls = [item for item in response.output if item.type == "function_call"]

        # No tool call means the agent has produced its final research notes.
        if not tool_calls:
            research_result = response.output_text.strip()

            if not research_result:
                raise RuntimeError(f"Research agent returned no result for step {step.id!r}.")

            if verbose:
                print(f"[{step.agent.upper()}] Research step {step.id} completed.")
                print(f"[{step.agent.upper()}] Total model calls: {model_calls}")
                print(f"[{step.agent.upper()}] Total tool rounds: {tool_rounds}")
                print(f"[{step.agent.upper()}] Total tool calls: {total_tool_calls}")
                print(f"[{step.agent.upper()}] Tool usage counts: {tool_counts}")
            return research_result

        for tool_call in tool_calls:
            total_tool_calls += 1
            tool_counts[tool_call.name] = tool_counts.get(tool_call.name, 0) + 1

            if verbose:
                print(
                    f"[{step.agent.upper()}] Tool call #{total_tool_calls}: "
                    f"{tool_call.name}, Arguments: {tool_call.arguments}"
                )

            try:
                arguments = json.loads(tool_call.arguments)

                result = execute_tool(
                    name=tool_call.name,
                    arguments=arguments,
                )

                if isinstance(result, list):
                    tool_errors = [
                        str(item["error"])
                        for item in result
                        if isinstance(item, dict) and "error" in item
                    ]
                    if tool_errors:
                        raise RuntimeError("; ".join(tool_errors))
                    result_summary = f"items={len(result)}"
                else:
                    result_summary = f"type={type(result).__name__}"

                tool_output = {
                    "ok": True,
                    "result": result,
                }

            except Exception as exc:  # noqa: BLE001
                tool_output = {
                    "ok": False,
                    "error": str(exc),
                }
                result_summary = f"error={type(exc).__name__}: {exc}"

            if verbose:
                print(f"[{step.agent.upper()}] Tool result #{total_tool_calls}: {result_summary}")

            input_items.append(
                {
                    "type": "function_call_output",
                    "call_id": tool_call.call_id,
                    "output": json.dumps(
                        tool_output,
                        ensure_ascii=False,
                    ),
                }
            )

        tool_rounds += 1

## 2.4 Writer Agent

In [ ]:
WRITER_INSTRUCTIONS = """
You are the Writer Agent in a multi-agent deep research system.

Execute exactly one writing step assigned by the Planner Agent.

The supplied execution history may contain:

- Research notes from one or more research steps
- An earlier draft
- Editorial feedback

Writing requirements:

- Follow the supplied task exactly.
- Use only the information contained in the execution history.
- Do not invent facts, sources, titles, dates, or URLs.
- Preserve source URLs exactly as provided.
- Clearly distinguish established findings from uncertain conclusions.
- Synthesize overlapping research rather than repeating it.
- Resolve minor inconsistencies when the supplied evidence allows it.
- Preserve important limitations and disagreements between sources.
- Use descriptive headings and a logical narrative structure.
- Include citations close to the claims they support.
- Avoid discussing the internal agent workflow in the report.
- Do not describe an arXiv preprint as peer reviewed unless the evidence says so.

If the task requests an initial draft:

- Synthesize the supplied research results into a coherent report.
- Ensure the report addresses the original user request.

If the task requests a revision:

- Use both the earlier draft and the editorial feedback.
- Address all required editorial changes.
- Return the complete revised report, not a list of modifications.
- When report_requirements are provided, follow their word limits.
- Use one top-level Markdown title and second-level section headings.
- Include ## Abstract, ## Executive Summary, and ## References sections.
- Deduplicate references and preserve available source metadata.
- Do not include a change summary or editorial commentary.

Return only the requested report content in Markdown.
Do not include a preface, afterword, change summary, follow-up question,
offer to continue, or any conversational text addressed to the user.
When producing the final report, end the document with the References section.
""".strip()

In [ ]:
def writer_agent(
    user_request: str,
    step: PlanStep,
    history: list[dict[str, Any]],
    model: str = MODEL_NAME,
    report_requirements: dict[str, int] | None = None,
    verbose: bool = True,
) -> str:
    if step.agent != "writer_agent":
        raise ValueError(
            f"Task step {step.id!r} is assigned to {step.agent!r}, not 'writer_agent'."
        )

    if verbose:
        print(f"[{step.agent.upper()}] Starting step {step.id}")
        print(f"[{step.agent.upper()}] Task: {step.task}")
        print(f"[{step.agent.upper()}] Context steps: {len(history)}")
        print(f"[{step.agent.upper()}] Model call #1")

    writer_request = {
        "user_request": user_request,
        "task_step": step.model_dump(mode="json"),
        "execution_history": history,
        "report_requirements": report_requirements,
    }

    response = client.responses.create(
        model=model,
        instructions=WRITER_INSTRUCTIONS,
        input=json.dumps(
            writer_request,
            ensure_ascii=False,
        ),
    )

    result = response.output_text.strip()

    if not result:
        raise RuntimeError(f"Writer agent returned no result for step {step.id!r}.")

    if verbose:
        print(f"[{step.agent.upper()}] Writing step {step.id} completed.")
        print(f"[{step.agent.upper()}] Output characters: {len(result)}")

    return result

## 2.5 Editor Agent

In [ ]:
EDITOR_INSTRUCTIONS = """
You are the Editor Agent in a multi-agent deep research system.

Critically review exactly one draft according to the step assigned by the
Planner Agent.

Review requirements:

- Evaluate whether the draft addresses the original user request.
- Evaluate factual support using only the supplied content.
- Identify unsupported or overstated claims.
- Check whether citations are placed near the claims they support.
- Identify missing citations, suspicious citations, and broken logical links.
- Check coverage, structure, clarity, concision, and internal consistency.
- Identify important research findings that were omitted or distorted.
- Distinguish required corrections from optional improvements.
- Do not invent facts or sources.
- Do not rewrite the entire report.
- Make every criticism specific and actionable.
- Keep the review concise and prioritize no more than ten required changes.

Return the review in Markdown using this structure:

## Verdict

Use either APPROVE or REVISE and provide a short explanation.

## Strengths

List the strongest aspects of the draft.

## Required Changes

List factual, structural, or coverage problems that must be fixed.

## Citation Issues

List unsupported claims and citation problems.

## Optional Improvements

List non-essential improvements.

Return only the structured editorial review. Do not include a rewritten
report or conversational text.

Your review will be consumed by the Writer Agent during revision.
""".strip()

In [ ]:
def editor_agent(
    user_request: str,
    step: PlanStep,
    history: list[dict[str, Any]],
    model: str = MODEL_NAME,
    verbose: bool = True,
) -> str:
    if step.agent != "editor_agent":
        raise ValueError(
            f"Task step {step.id!r} is assigned to {step.agent!r}, not 'editor_agent'."
        )

    if verbose:
        print(f"[{step.agent.upper()}] Starting step {step.id}")
        print(f"[{step.agent.upper()}] Task: {step.task}")
        print(f"[{step.agent.upper()}] Context steps: {len(history)}")
        print(f"[{step.agent.upper()}] Model call #1")

    editor_request = {
        "user_request": user_request,
        "task_step": step.model_dump(mode="json"),
        "execution_history": history,
    }

    response = client.responses.create(
        model=model,
        instructions=EDITOR_INSTRUCTIONS,
        input=json.dumps(
            editor_request,
            ensure_ascii=False,
        ),
    )

    result = response.output_text.strip()

    if not result:
        raise RuntimeError(f"Editor agent returned no result for step {step.id!r}.")

    if verbose:
        print(f"[{step.agent.upper()}] Editing step {step.id} completed.")
        print(f"[{step.agent.upper()}] Output characters: {len(result)}")

    return result

## 2.6 Static Executor Baseline

In [ ]:
AGENT_REGISTRY = {
    "research_agent": research_agent,
    "writer_agent": writer_agent,
    "editor_agent": editor_agent,
}


def executor_agent(
    user_request: str,
    plan: TaskPlan,
    model: str = MODEL_NAME,
    writer_requirements: dict[str, int] | None = None,
    verbose: bool = True,
) -> list[dict[str, Any]]:
    validate_task_plan(plan, max_steps=len(plan.steps))

    history: list[dict[str, Any]] = []

    if verbose:
        print("[EXECUTOR_AGENT] Starting plan execution.")
        print(f"[EXECUTOR_AGENT] Goal: {plan.goal}")
        print(f"[EXECUTOR_AGENT] Total steps: {len(plan.steps)}")

    for position, step in enumerate(plan.steps, start=1):
        agent_function = AGENT_REGISTRY.get(step.agent)

        if verbose:
            print(
                f"[EXECUTOR_AGENT] Dispatching step {position}/{len(plan.steps)} to {step.agent}."
            )

        try:
            if agent_function is None:
                raise ValueError(f"Unsupported agent: {step.agent!r}")

            agent_arguments = {
                "user_request": user_request,
                "step": step,
                "history": history,
                "model": model,
                "verbose": verbose,
            }

            if step.agent == "writer_agent":
                agent_arguments["report_requirements"] = writer_requirements

            result = agent_function(**agent_arguments)
        except Exception as exc:  # noqa: BLE001
            error = f"{type(exc).__name__}: {exc}"
            history.append(
                {
                    "step_id": step.id,
                    "agent": step.agent,
                    "task": step.task,
                    "status": "failed",
                    "result": None,
                    "error": error,
                }
            )

            if verbose:
                print(f"[EXECUTOR_AGENT] Step {step.id} failed: {error}")
                print("[EXECUTOR_AGENT] Plan execution stopped.")
            break

        history.append(
            {
                "step_id": step.id,
                "agent": step.agent,
                "task": step.task,
                "status": "completed",
                "result": result,
                "error": None,
            }
        )

        if verbose:
            print(f"[EXECUTOR_AGENT] Step {step.id} completed.")

    if verbose:
        completed_steps = sum(item["status"] == "completed" for item in history)
        failed_steps = sum(item["status"] == "failed" for item in history)
        if failed_steps == 0:
            print("[EXECUTOR_AGENT] Plan execution completed.")
        print(f"[EXECUTOR_AGENT] Completed steps: {completed_steps}")
        print(f"[EXECUTOR_AGENT] Failed steps: {failed_steps}")

    return history

## 2.7 V1 Entry Point and Quality Gate

This entry point remains available as a static baseline. It is intentionally
not invoked in V2; the future adaptive Supervisor will replace its planner and
executor path.

In [ ]:
def validate_final_report(
    report: str,
    min_words: int = 1200,
    max_words: int = 1800,
) -> int:
    """Validate deterministic publication requirements and return the word count."""

    if min_words < 1 or max_words < min_words:
        raise ValueError("Report word limits are invalid.")

    report = report.strip()

    if not report:
        raise RuntimeError("Final report is empty.")

    if re.search(r"(?m)^#\s+\S", report) is None:
        raise RuntimeError("Final report must contain a top-level Markdown title.")

    if re.search(r"(?im)^##\s+(references|sources)\s*$", report) is None:
        raise RuntimeError("Final report must contain a References or Sources section.")

    if re.search(r"https?://", report) is None:
        raise RuntimeError("Final report must contain source URLs.")

    word_count = len(report.split())

    if not min_words <= word_count <= max_words:
        raise RuntimeError(
            f"Final report has {word_count} words; expected {min_words} to {max_words}."
        )

    return word_count


def run_deep_research(
    topic: str,
    output_path: str | Path = "final_report.md",
    model: str = MODEL_NAME,
    max_steps: int = 4,
    min_report_words: int = 1200,
    max_report_words: int = 1800,
    verbose: bool = True,
) -> str:
    """Plan, execute, validate, and save one deep research report."""

    if min_report_words < 1 or max_report_words < min_report_words:
        raise ValueError("Report word limits are invalid.")

    writer_requirements = {
        "min_words": min_report_words,
        "max_words": max_report_words,
    }

    plan = planner_agent(
        topic=topic,
        model=model,
        max_steps=max_steps,
        verbose=verbose,
    )
    history = executor_agent(
        user_request=topic,
        plan=plan,
        model=model,
        writer_requirements=writer_requirements,
        verbose=verbose,
    )

    failed_steps = [item for item in history if item["status"] == "failed"]

    if failed_steps:
        failed_step = failed_steps[0]
        raise RuntimeError(
            f"Deep research failed at step {failed_step['step_id']}: {failed_step['error']}"
        )

    if len(history) != len(plan.steps):
        raise RuntimeError("Executor did not complete every planned step.")

    final_step = history[-1]

    if final_step["agent"] != "writer_agent":
        raise RuntimeError("The final completed step must use writer_agent.")

    final_report = final_step["result"]

    if not isinstance(final_report, str):
        raise TypeError("Final writer result must be a string.")

    final_report = final_report.strip()
    word_count = validate_final_report(
        final_report,
        min_words=min_report_words,
        max_words=max_report_words,
    )

    report_path = Path(output_path)
    report_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.write_text(f"{final_report}\n", encoding="utf-8")

    if verbose:
        print(f"[DEEP_RESEARCH] Final report validated: words={word_count}")
        print(f"[DEEP_RESEARCH] Final report saved to: {report_path}")

    return final_report

# 3. Adaptive Supervisor

The next development stage will consume `ResearchBrief` and create focused,
observation-driven research tasks.